<a href="https://colab.research.google.com/github/Neykone/ai-student-study-assistant/blob/main/Project_Lab8_Data_and_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project — Lab 8: data, evaluation set and baseline

**AI Student Study Assistant** · Kartit Aymane · KONE Cheick Aboubacar

Goal of this session (from the project brief): have the **data in hand**, build the **non-LLM part**
(PDF extraction + topic classifier), write an **evaluation set of 15–30 cases**, and measure a **baseline**.
No LLM pipeline yet — that is Lab 9.

Run the cells in order. Cells marked **TODO** need your input.

## Step 0 — Setup

In [ ]:
%pip install -q -U pypdf scikit-learn pandas google-genai
print("done — if Colab asks you to restart the session, do it, then re-run this cell.")

done — if Colab asks you to restart the session, do it, then re-run this cell.


In [ ]:
#@title ⚙️ Toolbox (copied from Lab 6: cached Gemini calls + rate limiter) { display-mode: "form" }
import json, re, time, hashlib, pathlib
import numpy as np, pandas as pd

CALL_LOG = []
MODEL_FAST = "gemini-3.5-flash-lite"   # vérifier le nom du modèle le jour du lab

_client = None
def _gemini():
    global _client
    if _client is None:
        from google import genai
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        assert key, "No GEMINI_API_KEY in Colab Secrets — see Lab 1, Step 0."
        _client = genai.Client(api_key=key.strip())
    return _client

CACHE = pathlib.Path("llm_cache"); CACHE.mkdir(exist_ok=True)
_last = [0.0]
MIN_INTERVAL = 4.2      # le niveau gratuit autorise ~15 requêtes par minute

def call_model(prompt, temperature=0.0, max_tokens=900, retries=4, use_cache=True):
    """Envoie un prompt, renvoie {"text", "latency_s", "tokens_in", "tokens_out", "cached"}.
    Chaque réponse est gardée sur disque : relancer le même prompt ne coûte rien."""
    key = hashlib.sha256(json.dumps([MODEL_FAST, prompt, temperature, max_tokens]).encode()).hexdigest()[:24]
    f = CACHE / f"{key}.json"
    if use_cache and f.exists():
        d = json.loads(f.read_text()); d["cached"] = True
        CALL_LOG.append({k: d[k] for k in ("latency_s", "tokens_in", "tokens_out", "cached")})
        return d
    from google.genai import types
    delay = 2.0
    for _ in range(retries):
        wait = MIN_INTERVAL - (time.time() - _last[0])
        if wait > 0: time.sleep(wait)
        try:
            t0 = time.time()
            r = _gemini().models.generate_content(
                model=MODEL_FAST, contents=prompt,
                config=types.GenerateContentConfig(temperature=temperature, max_output_tokens=max_tokens))
            _last[0] = time.time()
            u = getattr(r, "usage_metadata", None)
            d = {"text": (r.text or "").strip(), "latency_s": round(time.time() - t0, 2),
                 "tokens_in": getattr(u, "prompt_token_count", 0) or 0,
                 "tokens_out": getattr(u, "candidates_token_count", 0) or 0, "cached": False}
            if not d["text"]:
                raise RuntimeError("empty answer — raise max_tokens and try again")
            f.write_text(json.dumps(d))
            CALL_LOG.append({k: d[k] for k in ("latency_s", "tokens_in", "tokens_out", "cached")})
            return d
        except Exception as e:
            if "404" in str(e) or "NOT_FOUND" in str(e):
                raise RuntimeError(f"Model {MODEL_FAST!r} does not exist for this key — update MODEL_FAST. | {e}")
            _last[0] = time.time()
            print(f"  ⏳ {str(e)[:120]} — waiting {delay:.0f}s"); time.sleep(delay); delay *= 2
    raise RuntimeError("four attempts failed — check your key and your quota")

# ---------------------------------------------------------------- lecture des PDF (Lab 6)
def extract_pdf_pages(path):
    """Une chaîne par page, espaces normalisés, sauts de ligne gardés."""
    from pypdf import PdfReader
    out = []
    for p in PdfReader(str(path)).pages:
        t = p.extract_text() or ""
        out.append("\n".join(re.sub(r"[ \t\xa0]+", " ", l).strip() for l in t.split("\n")))
    return out

print("✅ toolbox loaded: call_model (cached), extract_pdf_pages")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


✅ toolbox loaded: call_model (cached), extract_pdf_pages


In [ ]:
# Vérifie que Gemini répond (1 vrai appel, puis mis en cache)
r = call_model("Reply with the single word OK.")
print(r["text"], "|", r["latency_s"], "s | cached:", r["cached"])

OK. | 4.87 s | cached: False


## Part A — The courses (PDFs), one folder per topic

Upload your PDFs into a folder named after their topic, e.g. `courses/NoSQL/cours1.pdf`,
`courses/MachineLearning/chap2.pdf`. **The folder name is the topic label.**

**TODO:** decide the 3–5 topics with your teammate, then upload the PDFs.

In [ ]:
import pathlib, shutil
from google.colab import files

TOPICS = ["MachineLearning", "NoSQL", "JEE"]    # TODO: vos thèmes (3 à 5), sans espaces

for t in TOPICS:
    pathlib.Path("courses", t).mkdir(parents=True, exist_ok=True)

for t in TOPICS:
    if not list(pathlib.Path("courses", t).glob("*.pdf")):
        print(f"Select the PDFs for topic {t!r} (several at once), or Cancel to skip:")
        try:
            up = files.upload()
            for name in up:
                shutil.move(name, pathlib.Path("courses", t, name))
        except Exception:
            pass

PDFS = {t: sorted(pathlib.Path("courses", t).glob("*.pdf")) for t in TOPICS}
for t, ps in PDFS.items():
    print(f"{t:<16} {len(ps)} PDF(s): {[p.name for p in ps]}")

Select the PDFs for topic 'MachineLearning' (several at once), or Cancel to skip:


Saving Decision_Trees_and_Ensemble_Learning_UIR_Revised.pdf to Decision_Trees_and_Ensemble_Learning_UIR_Revised.pdf
Saving Lecture 4 GDA.pdf to Lecture 4 GDA.pdf
Saving Machine Learning - Intro to ML.pdf to Machine Learning - Intro to ML.pdf
Saving Machine Learning - Linear classification.pdf to Machine Learning - Linear classification.pdf
Saving Machine Learning - Linear Regression and gradient descent (1).pdf to Machine Learning - Linear Regression and gradient descent (1).pdf
Select the PDFs for topic 'NoSQL' (several at once), or Cancel to skip:


Saving NoSQL_Lecture_1 _Students.pdf to NoSQL_Lecture_1 _Students.pdf
Saving NoSQl_Lecture_2_Students.pdf to NoSQl_Lecture_2_Students.pdf
Saving NoSQl_Lecture_3_Students.pdf to NoSQl_Lecture_3_Students.pdf
Saving NoSQl_Lecture_4_Students.pdf to NoSQl_Lecture_4_Students.pdf
Saving NoSQl_Lecture_5_Students.pdf to NoSQl_Lecture_5_Students.pdf
Saving NoSQl_Lecture_6_Students.pdf to NoSQl_Lecture_6_Students.pdf
Saving NoSQl_Lecture_7_Students.pdf to NoSQl_Lecture_7_Students.pdf
Select the PDFs for topic 'JEE' (several at once), or Cancel to skip:


Saving chap0.pdf to chap0.pdf
Saving chap1.pdf to chap1.pdf
Saving chap2_part1-handout.pdf to chap2_part1-handout.pdf
Saving chap3_handout.pdf to chap3_handout.pdf
Saving chap4_handout.pdf to chap4_handout.pdf
Saving chap5_handout.pdf to chap5_handout.pdf
Saving chap6_handout.pdf to chap6_handout.pdf
Saving chap7_handout.pdf to chap7_handout.pdf
MachineLearning  5 PDF(s): ['Decision_Trees_and_Ensemble_Learning_UIR_Revised.pdf', 'Lecture 4 GDA.pdf', 'Machine Learning - Intro to ML.pdf', 'Machine Learning - Linear Regression and gradient descent (1).pdf', 'Machine Learning - Linear classification.pdf']
NoSQL            7 PDF(s): ['NoSQL_Lecture_1 _Students.pdf', 'NoSQl_Lecture_2_Students.pdf', 'NoSQl_Lecture_3_Students.pdf', 'NoSQl_Lecture_4_Students.pdf', 'NoSQl_Lecture_5_Students.pdf', 'NoSQl_Lecture_6_Students.pdf', 'NoSQl_Lecture_7_Students.pdf']
JEE              8 PDF(s): ['chap0.pdf', 'chap1.pdf', 'chap2_part1-handout.pdf', 'chap3_handout.pdf', 'chap4_handout.pdf', 'chap5_handout.p

In [ ]:
# Extraction du texte, une ligne par page, avec sa provenance (topic, document, page)
rows = []
for t, ps in PDFS.items():
    for p in ps:
        for i, text in enumerate(extract_pdf_pages(p), start=1):
            rows.append({"topic": t, "doc": p.stem, "page": i, "text": text, "words": len(text.split())})
pages = pd.DataFrame(rows)

print(pages.groupby("topic").agg(docs=("doc", "nunique"), pages=("page", "count"), words=("words", "sum")))
assert len(pages) > 0, "No text extracted — did you upload the PDFs?"
pages.to_csv("pages.csv", index=False)
print("\n✅ pages.csv written —", len(pages), "pages")

                 docs  pages  words
topic                              
JEE                 8    135  21373
MachineLearning     5    258  10031
NoSQL               7    271  17969

✅ pages.csv written — 664 pages


In [ ]:
# A.1 — Regarde une page de chaque thème : y a-t-il un en-tête / pied de page qui se répète ?
for t in TOPICS:
    sub = pages[pages.topic == t]
    if len(sub):
        print("=" * 70, "\n", t, "—", sub.iloc[0]["doc"], "page 1\n")
        print(sub.iloc[0]["text"][:400], "…\n")

 MachineLearning — Decision_Trees_and_Ensemble_Learning_UIR_Revised page 1

Decision T rees
and Ensemble Learning
Learning to predict through simple questions
Prof. Y ouness Moukafih
International University of Rabat
 …

 NoSQL — NoSQL_Lecture_1 _Students page 1

No SQL
Module Instructor : Anas Knari (anas.knari@uir.ac.ma)
Academic Year : 2026-2027 …

 JEE — chap0 page 1

Chapter 0: Advanced Web Development
Course presentation
Saad Noufel
International University of Rabat
Academic year 2026–2027 …



## Part B — The evaluation set: student questions labelled by topic

This is the **deliverable of the session**: 15–30 cases minimum, we aim for **10–15 per topic**.

Rules (from the brief): write the question **as a student would type it**, do not copy sentences
from the PDFs, and label the topic **before** running any model. Mix French and English.

**TODO:** fill `questions.csv` (columns: `question`, `topic`). Two ways:
- edit the Python list below, or
- write it in Google Sheets, export as CSV and upload it.

In [ ]:
QUESTIONS = [
    ("Comment stocker une valeur avec une clé dans Redis ?",                 "NoSQL"),
    ("What is the difference between a document database and a table?",    "NoSQL"),
    ("Pourquoi mon modèle a un score parfait en entraînement et mauvais en test ?", "MachineLearning"),
    ("How do I choose between precision and recall?",                       "MachineLearning"),
    ("C'est quoi la différence entre une servlet et une JSP ?",             "JEE"),
    ("How does a session scope differ from a request scope?",              "JEE"),

    ("Comment mettre à jour un champ dans un document MongoDB ?",         "NoSQL"),
    ("Quelle est la différence entre une base orientée document et une base orientée graphe ?", "NoSQL"),
    ("Quand utiliser Redis plutôt que MongoDB ?",                         "NoSQL"),
    ("Peut-on faire des jointures dans une base orientée document ?",    "NoSQL"),
    ("Quelle est la différence entre une collection et un document ?",   "NoSQL"),

    ("What's the difference between bagging and boosting?",               "MachineLearning"),
    ("How do I handle an imbalanced dataset?",                            "MachineLearning"),
    ("Quelle métrique utiliser quand les classes ont un coût de mauvaise prédiction différent ?", "MachineLearning"),
    ("Comment éviter le surapprentissage ?",                              "MachineLearning"),
    ("Pourquoi la normalisation des features est importante ?",           "MachineLearning"),

    ("What's the lifecycle of a servlet?",                                "JEE"),
    ("Comment gérer l'authentification et les autorisations en JEE ?",    "JEE"),
    ("Quelle annotation pour injecter un EntityManager ?",                "JEE"),
    ("Quelle est la différence entre les méthodes GET et POST ?",         "JEE"),
    ("Comment faire communiquer deux servlets entre elles ?",             "JEE"),
    ("Comment fonctionne le sharding dans MongoDB ?",                    "NoSQL"),
    ("What is eventual consistency?",                                    "NoSQL"),
    ("Comment définir un TTL sur une clé Redis ?",                       "NoSQL"),
    ("Comment choisir le learning rate pour la descente de gradient ?",  "MachineLearning"),
    ("What does a random forest do differently from a single decision tree?", "MachineLearning"),
    ("C'est quoi la différence entre régression linéaire et classification ?", "MachineLearning"),
    ("Comment rediriger vers une autre page depuis une servlet ?",       "JEE"),
    ("What is the role of web.xml in a Tomcat project?",                 "JEE"),
    ("Comment partager un objet entre plusieurs pages JSP ?",            "JEE"),
]

questions = pd.DataFrame(QUESTIONS, columns=["question", "topic"])

# Si vous avez préparé un CSV à la place, décommentez :
# from google.colab import files; files.upload(); questions = pd.read_csv("questions.csv")

bad = set(questions.topic) - set(TOPICS)
assert not bad, f"Unknown topics in the questions: {bad}"
assert questions.question.is_unique, "Duplicate questions"
print(questions.topic.value_counts(), "\n")
print(f"{len(questions)} questions — target: at least 15, ideally 10–15 per topic")
questions.to_csv("questions.csv", index=False)

topic
NoSQL              10
MachineLearning    10
JEE                10
Name: count, dtype: int64 

30 questions — target: at least 15, ideally 10–15 per topic


In [ ]:
# B.1 — Séparer en deux : TRAIN (pour apprendre) et TEST (pour mesurer). Jamais les mêmes questions.
from sklearn.model_selection import train_test_split

train, test = train_test_split(questions, test_size=0.4, random_state=0, stratify=questions.topic)
print(f"train: {len(train)}   test: {len(test)}")
print("\ntest set by topic:\n", test.topic.value_counts())
test.to_csv("eval_set.csv", index=False)
print("\n✅ eval_set.csv written — this is the file you keep for Lab 9 and the defence")

train: 18   test: 12

test set by topic:
 topic
JEE                4
NoSQL              4
MachineLearning    4
Name: count, dtype: int64

✅ eval_set.csv written — this is the file you keep for Lab 9 and the defence


## Part C — The baseline, and the first non-LLM model

Two numbers before any LLM:
1. **Baseline** — always predict the most frequent topic of the training set. Any model must beat it.
2. **Naive Bayes** on word counts (our non-LLM component from the charter).

Both are scored on the **same test set** with the same metric: **accuracy** = share of test questions whose predicted topic is correct.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1) Baseline : le thème le plus fréquent dans TRAIN
most_frequent = train.topic.value_counts().idxmax()
baseline_pred = [most_frequent] * len(test)
acc_baseline = accuracy_score(test.topic, baseline_pred)
print(f"baseline (always {most_frequent!r}): accuracy = {acc_baseline:.0%}")

# 2) Naive Bayes : compte les mots de la question, apprend quels mots vont avec quel thème
nb = make_pipeline(CountVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2)),
                   MultinomialNB())
nb.fit(train.question, train.topic)
nb_pred = nb.predict(test.question)
acc_nb = accuracy_score(test.topic, nb_pred)
print(f"Naive Bayes: accuracy = {acc_nb:.0%}   ({(np.array(nb_pred) == test.topic.values).sum()}/{len(test)})")

print("\n", classification_report(test.topic, nb_pred, zero_division=0))
print("confusion matrix (rows = true topic, columns = predicted):")
print(pd.DataFrame(confusion_matrix(test.topic, nb_pred, labels=TOPICS), index=TOPICS, columns=TOPICS))

baseline (always 'NoSQL'): accuracy = 33%
Naive Bayes: accuracy = 42%   (5/12)

                  precision    recall  f1-score   support

            JEE       0.00      0.00      0.00         4
MachineLearning       0.50      0.25      0.33         4
          NoSQL       0.40      1.00      0.57         4

       accuracy                           0.42        12
      macro avg       0.30      0.42      0.30        12
   weighted avg       0.30      0.42      0.30        12

confusion matrix (rows = true topic, columns = predicted):
                 MachineLearning  NoSQL  JEE
MachineLearning                1      3    0
NoSQL                          0      4    0
JEE                            1      3    0


In [ ]:
# C.1 — Les erreurs, une par une :  ce que le prof veut voir, pas seulement le pourcentage
errors = test.assign(predicted=nb_pred)
errors = errors[errors.topic != errors.predicted]
print(f"{len(errors)} error(s) on {len(test)} test questions\n")
for _, r in errors.iterrows():
    print(f"  true={r.topic:<16} predicted={r.predicted:<16} {r.question[:70]}")

7 error(s) on 12 test questions

  true=JEE              predicted=MachineLearning  Comment gérer l'authentification et les autorisations en JEE ?
  true=MachineLearning  predicted=NoSQL            Pourquoi la normalisation des features est importante ?
  true=JEE              predicted=NoSQL            C'est quoi la différence entre une servlet et une JSP ?
  true=JEE              predicted=NoSQL            Quelle est la différence entre les méthodes GET et POST ?
  true=MachineLearning  predicted=NoSQL            Quelle métrique utiliser quand les classes ont un coût de mauvaise pré
  true=MachineLearning  predicted=NoSQL            Pourquoi mon modèle a un score parfait en entraînement et mauvais en t
  true=JEE              predicted=NoSQL            Comment rediriger vers une autre page depuis une servlet ?


In [ ]:
# C.2 — Essayer le classifieur sur une question nouvelle
for q in ["Comment faire une jointure entre deux collections ?",
          "What is gradient descent?",
          "How do I configure a filter in web.xml?"]:
    print(f"{nb.predict([q])[0]:<16} ← {q}")

JEE              ← Comment faire une jointure entre deux collections ?
NoSQL            ← What is gradient descent?
MachineLearning  ← How do I configure a filter in web.xml?


In [ ]:
# C.3 — Deuxième version : apprendre AUSSI sur les pages des PDF (déjà étiquetées par leur dossier).
# Quelques dizaines de questions, c'est peu ; des centaines de pages de cours, c'est beaucoup plus de mots par thème.
train_text  = pd.concat([train.question, pages.text], ignore_index=True)
train_label = pd.concat([train.topic,    pages.topic], ignore_index=True)

nb_pages = make_pipeline(CountVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2), min_df=1),
                         MultinomialNB())
nb_pages.fit(train_text, train_label)
pred_pages = nb_pages.predict(test.question)
acc_nb_pages = accuracy_score(test.topic, pred_pages)

results = pd.DataFrame([
    {"model": f"baseline (always {most_frequent})", "trained on": "—",                "accuracy": acc_baseline},
    {"model": "Naive Bayes",                        "trained on": "questions only",   "accuracy": acc_nb},
    {"model": "Naive Bayes",                        "trained on": "questions + PDF pages", "accuracy": acc_nb_pages},
])
print(results.to_string(index=False))
print(f"\n(test set: {len(test)} questions — one question = {1/len(test):.0%} of accuracy, so small gaps are noise)")

                  model            trained on  accuracy
baseline (always NoSQL)                     —  0.333333
            Naive Bayes        questions only  0.416667
            Naive Bayes questions + PDF pages  0.833333

(test set: 12 questions — one question = 8% of accuracy, so small gaps are noise)


## Part D — Session notes (write before leaving)

Fill the three texts; they go straight into the Lab 9 notebook and the report.

In [ ]:


DATA_IN_HAND = """
20 lecture PDFs from our UIR instructors: Machine Learning (5 PDFs, 258 pages), NoSQL (7 PDFs, 271 pages), JEE (8 PDFs, 135 pages), 664 pages in total, topic = folder name. 30 student-style questions (10 per topic, French and English), drafted with an AI assistant, then checked and labelled by us before any model was run. Split: 18 train / 12 test.
"""

BASELINE_RESULT = """
On 12 test questions: baseline (always NoSQL) 33% (4/12), Naive Bayes on questions only 42% (5/12), Naive Bayes on questions + PDF pages 83% (10/12). Example error of the questions-only model: 'Comment rediriger vers une autre page depuis une servlet ?' (JEE) predicted NoSQL, likely because words like 'servlet' never appear in its 18 training questions.
"""



for n, t in [("DATA_IN_HAND", DATA_IN_HAND), ("BASELINE_RESULT", BASELINE_RESULT)]:
    assert "TODO" not in t and len(t.strip()) > 60, f"{n} is not written yet"
print("✅ notes complete")

✅ notes complete


In [ ]:
# Sauvegarde de la séance : télécharger ces fichiers et les garder dans le dépôt du projet
summary = {"topics": TOPICS, "n_pages": int(len(pages)), "n_questions": int(len(questions)),
           "n_train": int(len(train)), "n_test": int(len(test)),
           "baseline_accuracy": round(float(acc_baseline), 3), "naive_bayes_accuracy": round(float(acc_nb), 3),
           "naive_bayes_pages_accuracy": round(float(acc_nb_pages), 3),
           "notes": {"data": DATA_IN_HAND.strip(), "baseline": BASELINE_RESULT.strip()},
           "n_llm_calls": len(CALL_LOG), "created": time.strftime("%Y-%m-%d %H:%M")}
pathlib.Path("lab8_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\n📦 keep: pages.csv, questions.csv, eval_set.csv, lab8_summary.json, and this notebook")

{
  "topics": [
    "MachineLearning",
    "NoSQL",
    "JEE"
  ],
  "n_pages": 664,
  "n_questions": 30,
  "n_train": 18,
  "n_test": 12,
  "baseline_accuracy": 0.333,
  "naive_bayes_accuracy": 0.417,
  "naive_bayes_pages_accuracy": 0.833,
  "notes": {
    "data": "20 lecture PDFs from our UIR instructors: Machine Learning (5 PDFs, 258 pages), NoSQL (7 PDFs, 271 pages), JEE (8 PDFs, 135 pages), 664 pages in total, topic = folder name. 30 student-style questions (10 per topic, French and English), drafted with an AI assistant, then checked and labelled by us before any model was run. Split: 18 train / 12 test.",
    "baseline": "On 12 test questions: baseline (always NoSQL) 33% (4/12), Naive Bayes on questions only 42% (5/12), Naive Bayes on questions + PDF pages 83% (10/12). Example error of the questions-only model: 'Comment rediriger vers une autre page depuis une servlet ?' (JEE) predicted NoSQL, likely because words like 'servlet' never appear in its 18 training questions."
  },
 